In [1]:
import pandas as pd
from datasets import load_from_disk

ds = load_from_disk("data/ag_news")

train_df = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()

train_df["split"] = "train"
test_df["split"] = "test"

df = pd.concat(
    [train_df, test_df],
    ignore_index=True,
)

print(df.shape)
display(df.head())

(127600, 3)


,text,label,split
0,Wall St. Bears Claw Back Into the Black (Reute...,2,train
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2,train
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2,train
3,Iraq Halts Oil Exports from Main Southern Pipe...,2,train
4,"Oil prices soar to all-time record, posing new...",2,train


In [2]:
print("no.label:", df["label"].nunique())
print("label value:", sorted(df["label"].dropna().unique()))

print("sample numbers for each label:")
print(df["label"].value_counts(dropna=False).sort_index())

print("missing value:", df.isna().sum())
print("duplicate column:", df.duplicated().sum())

no.label: 4
label value: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
sample numbers for each label:
label
0    31900
1    31900
2    31900
3    31900
Name: count, dtype: int64
missing value: text     0
label    0
split    0
dtype: int64
duplicate column: 0


In [9]:
import json
import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch


OUTPUT_ROOT = Path(OUTPUT_ROOT)
FOLD_RESULT_CSV = Path(FOLD_RESULT_CSV)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError(
        "没有检测到 CUDA GPU，请先解决 GPU 环境问题。"
    )

# 读取已经保存的结果
if FOLD_RESULT_CSV.exists():
    all_fold_results = pd.read_csv(FOLD_RESULT_CSV)
else:
    all_fold_results = pd.DataFrame()


def save_fold_result(row):
    """保存一折结果，保留其他模型和折的记录。"""
    global all_fold_results

    if not all_fold_results.empty:
        same_fold = (
            all_fold_results["model"].eq(row["model"])
            & all_fold_results["fold"].astype(int).eq(row["fold"])
        )

        all_fold_results = (
            all_fold_results.loc[~same_fold].copy()
        )

    all_fold_results = pd.concat(
        [all_fold_results, pd.DataFrame([row])],
        ignore_index=True,
    )

    temporary_csv = FOLD_RESULT_CSV.with_suffix(".tmp")

    all_fold_results.to_csv(
        temporary_csv,
        index=False,
    )

    temporary_csv.replace(FOLD_RESULT_CSV)


for model_name in MODELS:
    safe_name = model_name.replace("/", "__")
    model_dir = OUTPUT_ROOT / safe_name
    model_dir.mkdir(parents=True, exist_ok=True)

    completed_folds = set()

    if not all_fold_results.empty:
        previous = all_fold_results.loc[
            all_fold_results["model"].eq(model_name)
        ]

        completed_folds = set(
            previous["fold"].astype(int)
        )

    # 如果 JSON 已保存，但 CSV 尚未更新，则补回 CSV
    for fold in range(1, N_SPLITS + 1):
        metrics_file = (
            model_dir / f"fold_{fold}" / "metrics.json"
        )

        if fold not in completed_folds and metrics_file.exists():
            with metrics_file.open(
                encoding="utf-8"
            ) as file:
                saved_row = json.load(file)

            save_fold_result(saved_row)
            completed_folds.add(fold)

    if set(range(1, N_SPLITS + 1)).issubset(completed_folds):
        print("Skipping completed model:", model_name)
        continue

    # 每个模型只对整个训练集分词一次
    tokenizer, tokenized_dataset, data_collator = (
        prepare_tokenized_dataset(
            model_name=model_name,
            training_df=training_df,
        )
    )

    for fold, (train_indices, eval_indices) in enumerate(
        fold_indices,
        start=1,
    ):
        if fold in completed_folds:
            print(f"Skipping: {model_name}, fold {fold}")
            continue

        print("\n" + "=" * 60)
        print("Model:", model_name)
        print(f"Fold: {fold}/{N_SPLITS}")

        fold_dir = model_dir / f"fold_{fold}"
        fold_dir.mkdir(parents=True, exist_ok=True)

        set_seed(TRAINING_SEED)

        train_dataset = tokenized_dataset.select(
            train_indices.tolist()
        )

        eval_dataset = tokenized_dataset.select(
            eval_indices.tolist()
        )

        # 每折重新加载预训练模型，独立微调
        model = (
            AutoModelForSequenceClassification.from_pretrained(
                model_name,
                num_labels=NUM_LABELS,
                problem_type="single_label_classification",
            )
        )

        training_args = TrainingArguments(
            output_dir=str(fold_dir),

            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            gradient_accumulation_steps=(
                GRADIENT_ACCUMULATION_STEPS
            ),

            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            optim=OPTIMIZER,
            lr_scheduler_type=LR_SCHEDULER_TYPE,
            warmup_ratio=WARMUP_RATIO,

            eval_strategy="no",
            save_strategy="no",
            logging_steps=100,

            seed=TRAINING_SEED,
            data_seed=TRAINING_SEED,

            fp16=FP16,
            bf16=BF16,
            report_to="none",
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            data_collator=data_collator,
            compute_metrics=compute_classification_metrics,
        )

        print("Training device:", trainer.args.device)

        # 1. 训练
        start_time = time.perf_counter()

        train_result = trainer.train()

        training_seconds = time.perf_counter() - start_time

        # 2. 评估当前折，不使用官方 test
        prediction = trainer.predict(eval_dataset)
        logits = prediction.predictions

        if isinstance(logits, tuple):
            logits = logits[0]

        # 3. 保存训练后的模型和 tokenizer
        saved_model_dir = fold_dir / "trained_model"

        trainer.save_model(str(saved_model_dir))
        tokenizer.save_pretrained(str(saved_model_dir))

        # 4. 保存预测及对应的文本 ID
        np.savez_compressed(
            fold_dir / "predictions.npz",
            document_ids=(
                training_df.iloc[eval_indices]["document_id"]
                .to_numpy(dtype=str)
            ),
            true_labels=prediction.label_ids,
            logits=logits,
            predicted_labels=np.argmax(logits, axis=1),
        )

        # 5. 保存参数和训练日志
        (fold_dir / "training_args.json").write_text(
            training_args.to_json_string(),
            encoding="utf-8",
        )

        trainer.state.save_to_json(
            str(fold_dir / "trainer_state.json")
        )

        # 6. 整理当前折的结果
        row = {
            "dataset": "AGNews",
            "model": model_name,
            "fold": fold,

            "accuracy": float(
                prediction.metrics["test_accuracy"]
            ),
            "macro_f1": float(
                prediction.metrics["test_macro_f1"]
            ),
            "micro_f1": float(
                prediction.metrics["test_micro_f1"]
            ),
            "eval_loss": float(
                prediction.metrics["test_loss"]
            ),
            "training_loss": float(
                train_result.training_loss
            ),

            "training_hours": training_seconds / 3600,
            "train_size": len(train_dataset),
            "eval_size": len(eval_dataset),

            "seed": TRAINING_SEED,
            "epochs": NUM_EPOCHS,
            "max_length": MAX_LENGTH,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "eval_batch_size": EVAL_BATCH_SIZE,
            "gradient_accumulation_steps": (
                GRADIENT_ACCUMULATION_STEPS
            ),
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
        }

        # JSON 写入成功后，这一折才有完成标记
        temporary_json = fold_dir / "metrics.tmp"

        temporary_json.write_text(
            json.dumps(row, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

        temporary_json.replace(
            fold_dir / "metrics.json"
        )

        save_fold_result(row)

        print(
            f"Saved fold {fold} | "
            f"Accuracy: {row['accuracy']:.4f} | "
            f"Macro F1: {row['macro_f1']:.4f} | "
            f"Training hours: {row['training_hours']:.2f}"
        )

        # 7. 释放当前折的资源
        del trainer, model, train_dataset, eval_dataset
        del prediction, logits, train_result

        gc.collect()
        torch.cuda.empty_cache()

    del tokenizer, tokenized_dataset, data_collator
    gc.collect()


print("\nResults saved to:", FOLD_RESULT_CSV)


Tokenizing google-bert/bert-base-uncased:   0%|          | 0/120000 [00:00<?, ? examples/s]


Model: google-bert/bert-base-uncased
Fold: 1/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecate

Training device: cuda:0


Step,Training Loss
100,0.653498
200,0.322130
300,0.329201
400,0.310326
500,0.303193
600,0.255901
700,0.299065
800,0.283553
900,0.306326
1000,0.266253


... (219 outputs left out)

Training device: cuda:0


Step,Training Loss
100,1.017001
200,0.609025
300,0.512170
400,0.509938
500,0.490705
600,0.420860
700,0.412296
800,0.448979
900,0.397039
1000,0.407046


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fold 5 | Accuracy: 0.9329 | Macro F1: 0.9328 | Training hours: 0.25

Results saved to: /home/amyliu/Desktop/GR/ACGNews/results/ag_news_5fold/ag_news_5fold_results.csv


In [10]:
from pathlib import Path
import pandas as pd

result_file = Path(
    "/home/amyliu/Desktop/GR/ACGNews/results/"
    "ag_news_5fold/ag_news_5fold_results.csv"
)

fold_results = pd.read_csv(result_file)

# 查看每个模型完成了多少折
display(
    fold_results.groupby("model", as_index=False)
    .agg(completed_folds=("fold", "nunique"))
)

# 查看每折的具体结果
display(
    fold_results[
        ["model", "fold", "accuracy", "macro_f1",
         "micro_f1", "training_hours"]
    ].sort_values(["model", "fold"])
)

,model,completed_folds
0,Lianglab/PharmBERT-uncased,5
1,ahmedrachid/FinancialBERT,5
2,allenai/scibert_scivocab_uncased,5
3,google-bert/bert-base-uncased,5
4,nlpaueb/legal-bert-base-uncased,5
5,recobo/agriculture-bert-uncased,5
6,recobo/chemical-bert-uncased,5


,model,fold,accuracy,macro_f1,micro_f1,training_hours
20,Lianglab/PharmBERT-uncased,1,0.945000,0.945011,0.945000,0.238318
21,Lianglab/PharmBERT-uncased,2,0.945542,0.945551,0.945542,0.238265
22,Lianglab/PharmBERT-uncased,3,0.946750,0.946740,0.946750,0.238333
23,Lianglab/PharmBERT-uncased,4,0.944167,0.944187,0.944167,0.238383
24,Lianglab/PharmBERT-uncased,5,0.945542,0.945564,0.945542,0.238429
15,ahmedrachid/FinancialBERT,1,0.927333,0.927361,0.927333,0.255633
16,ahmedrachid/FinancialBERT,2,0.926167,0.926040,0.926167,0.255797
17,ahmedrachid/FinancialBERT,3,0.928833,0.928764,0.928833,0.255841
18,ahmedrachid/FinancialBERT,4,0.929625,0.929619,0.929625,0.256018
19,ahmedrachid/FinancialBERT,5,0.926833,0.926815,0.926833,0.255980


In [11]:
real_model_ranking = (
    fold_results
    .groupby("model", as_index=False)
    .agg(
        completed_folds=("fold", "nunique"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        micro_f1_mean=("micro_f1", "mean"),
        micro_f1_std=("micro_f1", "std"),
        training_hours_total=("training_hours", "sum"),
    )
)

# 只对完成全部五折的模型排名
real_model_ranking = (
    real_model_ranking[
        real_model_ranking["completed_folds"] == 5
    ]
    .copy()
)

real_model_ranking["real_rank"] = (
    real_model_ranking["macro_f1_mean"]
    .rank(ascending=False, method="min")
    .astype(int)
)

real_model_ranking = (
    real_model_ranking
    .sort_values(["real_rank", "model"])
    .reset_index(drop=True)
)

# 把排名放到第一列
real_model_ranking = real_model_ranking[
    ["real_rank", "model"]
    + [
        col for col in real_model_ranking.columns
        if col not in ["real_rank", "model"]
    ]
]

display(real_model_ranking)

real_model_ranking.to_csv(
    result_file.parent / "ag_news_real_model_ranking.csv",
    index=False,
)

,real_rank,model,completed_folds,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,micro_f1_mean,micro_f1_std,training_hours_total
0,1,google-bert/bert-base-uncased,5,0.947375,0.001030,0.947388,0.001008,0.947375,0.001030,1.200185
1,2,Lianglab/PharmBERT-uncased,5,0.945400,0.000941,0.945411,0.000931,0.945400,0.000941,1.191728
2,3,allenai/scibert_scivocab_uncased,5,0.942642,0.002091,0.942653,0.002073,0.942642,0.002091,1.270295
3,4,recobo/agriculture-bert-uncased,5,0.942525,0.001233,0.942519,0.001233,0.942525,0.001233,1.241469
4,5,nlpaueb/legal-bert-base-uncased,5,0.941858,0.001328,0.941838,0.001302,0.941858,0.001328,1.247366
5,6,recobo/chemical-bert-uncased,5,0.932142,0.000756,0.932110,0.000744,0.932142,0.000756,1.241426
6,7,ahmedrachid/FinancialBERT,5,0.927758,0.001433,0.927720,0.001455,0.927758,0.001433,1.279269
